# 02 · Etiketler: 58 altın vaka ve rapordan türetilen yumuşak etiketler

**Bu notebook ne yapar?** Eğitimde kullanacağımız 12 bulgu etiketini hazırlar ve bu etiketlerin
ne kadar güvenilir olduğunu ölçer. Çıktısı `data/train_labels.csv` dosyasıdır; eğitim
notebook'u (05) bu dosyayı okur.

**Neden ayrı bir adım?** Yarışmanın asıl zorluğu görüntüde değil etikettedir:

| | vaka | 12 etiketi var | raporu var |
|---|---:|:---:|:---:|
| `train.csv` | 4.407 | **yalnızca 58** | hepsi |
| `test.csv` | gizli | — | **yok** |

* Etiketsiz 4.349 vaka için etiketleri raporlardan (çok dilli serbest metin) çıkarmak zorundayız.
* Test setinde rapor olmadığı için metin, tahmin anında kullanılamaz; yalnızca **eğitim hedefi**
  üretmeye yarar. Yani "metin + görüntü füzyonu" bu yarışmada mümkün değil.
* 58 altın vaka hem etiket kaynaklarını seçmek hem de modeli doğrulamak için tek gerçek ölçüdür;
  ama 58 vaka çok azdır, küçük farklar istatistiksel gürültüdür (aşağıda ölçüyoruz).

**Kullanılan modül:** `src/labels.py`

In [ ]:
# Proje kökünü (src/ klasörünü içeren yer) Python yoluna ekle: yerelde ve Kaggle'da çalışır.
import os, sys
from pathlib import Path

def add_project_root():
    cands = [Path.cwd(), Path.cwd().parent]
    if Path("/kaggle/input").exists():
        # os.walk + budama: yarışmanın DICOM klasörlerine girmez (rglob dakikalar sürüyordu)
        for d, dirs, files in os.walk("/kaggle/input"):
            dirs[:] = [x for x in dirs if x not in ("train_series", "test_series")]
            if Path(d).name == "src" and "__init__.py" in files:
                cands.append(Path(d).parent)
    for c in cands:
        if (c / "src" / "__init__.py").exists():
            sys.path.insert(0, str(c))
            return c
    raise FileNotFoundError("src/ bulunamadı")

PROJECT_ROOT = add_project_root()
print("Proje kökü:", PROJECT_ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import DATA_DIR, TARGET_COLS
from src import labels as L

pd.set_option("display.precision", 3)
gold = L.load_gold()
print(f"Altın vaka sayısı: {len(gold)}")

## 1. Etiket kaynakları

İki bağımsız, herkese açık kaynak kullanıyoruz (yarışma kuralları public harici veriye izin veriyor):

| ad | Kaggle dataset | dosya | nasıl üretilmiş |
|---|---|---|---|
| `steven_v4` | `stevenleehans/rsna-knee-llm-report-labels` | `llm_labels_v4_blend.csv` | LLM okuması + blend |
| `pilkwang_v2` | `pilkwang/rsna-knee-llm-labels` | `report_labels_v2.csv` | sözlük (lexicon) kuralları + LLM ikinci okuyucu |
| `opus45` | github.com/NTejas-1/RSNA-Knee-Abnormality-Detection (MIT) | `labels_v1_opus45.csv` | Claude Opus 4.5 (10-14'te eklendi) |
| `jev_hybrid` | `riadmohamed42/jev-knee-labels` | `HYBRID_labels_scores.csv` | steven + pilkwang + metin modeli karışımı (10-14'te eklendi) |

> Not: Repodaki eski `data/llm_labels_*.csv` dosyaları aslında `stevenleehans` setinin
> kopyalarıdır; `scripts/audit_labels.py` bunlar arasından en iyisini seçiyordu.

Önce her kaynağın 58 altın vakadaki ROC AUC'sine bakalım. Bu, "rapor okuyucusu radyoloğun
etiketini ne kadar iyi sıralıyor?" sorusunun cevabıdır.

In [ ]:
ALL_SOURCES = {**L.SOURCES_V1, **L.SOURCES_V2}
audit = L.audit_sources(ALL_SOURCES)
audit.style.background_gradient(axis=None, cmap="RdYlGn", vmin=0.65, vmax=1.0).format("{:.3f}")

**Okuma:** ACL, MCL, Baker's gibi raporda net yazılan bulgular ~0.95+; **Synovitis,
Fracture, Lateral OA, Contusion** ise zayıf. Bu bulgular raporlarda dolaylı ifade ediliyor
("kemik iliği ödemi", "sinovyal kalınlaşma" vb.) ve dil kapsamı eksik kalıyor.
Modelimizin tavanı büyük ölçüde bu satırlarla sınırlı.

## 2. Pozitiflik oranları: kaynaklar altın vakalarla uyuşuyor mu?

In [ ]:
src = {k: L.load_source(f) for k, f in ALL_SOURCES.items()}
prev = pd.DataFrame({"altın (58)": gold.mean()})
for k, df in src.items():
    prev[k + " (>0.5)"] = (df > 0.5).mean()
ax = prev.plot.bar(figsize=(13, 4), rot=30, title="Bulgu bazında pozitiflik oranı")
ax.set_ylabel("oran"); plt.tight_layout(); plt.show()
prev.T

**Okuma:** Rapor tabanlı etiketler Fracture, Synovitis ve Lateral Meniscus'u altın vakalara
göre belirgin biçimde **az** işaretliyor. AUC sıralamaya baktığı için bu kalibrasyon farkı
metriği doğrudan bozmaz, ama:
* eğitimde `pos_weight` gibi oran tabanlı ayarlar yanlı olur,
* doğrulamada etiketleri 0.5'te ikiye bölmek bu bulgularda pozitif sınıfı küçültür.

Bu yüzden yeni eğitimde sabit `pos_weight` kullanmıyoruz; yumuşak etiketle doğrudan BCE.

## 3. İki kaynak ne kadar birbirine benziyor?

In [ ]:
ng = src["steven_v4"].index.difference(gold.index)
corr = pd.Series({c: np.corrcoef(src["steven_v4"].loc[ng, c],
                                  src["pilkwang_v2"].reindex(ng)[c].fillna(0.5))[0, 1]
                  for c in TARGET_COLS}, name="korelasyon")
corr.to_frame().T

Korelasyon 0.81–0.98: kaynaklar büyük ölçüde aynı şeyi söylüyor, ama Contusion ve Fracture'da
ayrışıyorlar. İki bağımsız okuyucunun ortalaması, tek tek hatalarını kısmen sönümler.

## 4. Ortalama almak gerçekten işe yarıyor mu? (58 vakanın gürültüsü)

58 vakalık AUC'nin ne kadar oynak olduğunu **bootstrap** ile ölçüyoruz: vakaları tekrar
örnekleyip (iadeli) farkı 300 kez hesaplıyoruz. Güven aralığı sıfırı içeriyorsa fark
"gerçek" değildir.

In [ ]:
from sklearn.metrics import roc_auc_score

blend = (src["steven_v4"] + src["pilkwang_v2"].reindex(src["steven_v4"].index).fillna(src["steven_v4"])) / 2
def macro(idx, pred):
    return np.mean([roc_auc_score(gold.loc[idx, c], pred.loc[idx, c]) for c in TARGET_COLS])

rng = np.random.default_rng(0)
deltas = []
for _ in range(300):
    idx = rng.choice(gold.index, len(gold))
    try:
        deltas.append(macro(idx, blend) - macro(idx, src["steven_v4"]))
    except ValueError:  # örneklemde bir sınıf tek değer kalırsa atla
        pass
lo, hi = np.percentile(deltas, [5, 95])
print(f"blend - steven_v4 : ortalama {np.mean(deltas):+.4f}, %90 aralık [{lo:+.4f}, {hi:+.4f}]")
print(f"blend makro AUC (58): {macro(gold.index, blend):.4f}")

**Sonuç:** Fark gürültü bandının içinde. 58 vaka iki kaynağı ayırt etmeye yetmez; bu yüzden
"altın vakada en yüksek skoru alanı seç" yaklaşımı (eski `audit_labels.py`) aslında rastgele
seçimdir ve doğrulama setine sızıntı yaratır.

**Karar:** İki kaynağın eşit ağırlıklı ortalaması. Gerekçe istatistiksel değil yapısal:
bağımsız okuyucuları ortalamak varyansı düşürür ve hiçbir kaynağa altın vakalarla
"ayar" yapmamış oluruz.

## 5. Eğitim etiket tablosunu oluştur

* Etiketsiz vakalar: iki kaynağın ortalaması (0–1 arası yumuşak etiket).
* 58 altın vaka: radyolog etiketi (0/1), `is_gold=True`. Eğitimde bu vakalara 2× ağırlık
  verilir ve doğrulamada ayrıca raporlanır.

In [ ]:
train_labels = L.build_training_labels(L.SOURCES_V1)
out_path = DATA_DIR / "train_labels.csv"
train_labels.to_csv(out_path, index=False)
print(f"Kaydedildi: {out_path}  ->  {train_labels.shape}")
print(f"Altın vaka: {train_labels.is_gold.sum()}  |  boş değer: {train_labels[TARGET_COLS].isna().sum().sum()}")
train_labels.head()

In [ ]:
fig, ax = plt.subplots(figsize=(13, 3.5))
train_labels.loc[~train_labels.is_gold, TARGET_COLS].plot.box(ax=ax, rot=30)
ax.set_title("Yumuşak etiket dağılımı (etiketsiz 4.349 vaka)")
plt.tight_layout(); plt.show()

## 6. Etiket v2 (2026-10-14): üçüncü ve dördüncü okuyucu

Yarışma forumunda iki güçlü public kaynak bulundu: Claude Opus 4.5 etiketleri ve JEV HYBRID.
JEV'in içinde steven ve pilkwang zaten bulunduğu için pilkwang'ı ayrıca eklemiyoruz (çift sayım).
Aşağıda olası karışımların 58 altın vakadaki makro AUC'si:

In [ ]:
import itertools
names = ["steven_v4", "pilkwang_v2", "opus45", "jev_hybrid"]
mean_all = pd.concat([src[n] for n in names]).groupby(level=0).mean()
rows = []
for r in range(1, 5):
    for combo in itertools.combinations(names, r):
        b = sum(src[n].reindex(mean_all.index).fillna(mean_all) for n in combo) / len(combo)
        rows.append({"karışım": " + ".join(combo), "makro AUC": L.macro_auc(gold, b)["macro"]})
pd.DataFrame(rows).sort_values("makro AUC", ascending=False).head(8).round(4)

**Karar:** etiket v2 = steven_v4 + opus45 + jev_hybrid (0.9014). Farklar 58 vakanın gürültü
bandına yakın, ama üç bağımsız okuyucunun ortalaması yapısal olarak daha sağlam. Qwen-7B ile kendi
etiketlememiz (08) 0.797'de kaldı ve eklenmedi. Not: altın etiketler bazen raporla çelişiyor
(radyolog görüntüye bakmış), bu yüzden rapor tabanlı etiketlerin tavanı ~0.90.

In [ ]:
train_labels_v2 = L.build_training_labels(L.SOURCES_V2)
train_labels_v2.to_csv(DATA_DIR / "train_labels_v2.csv", index=False)
print("Kaydedildi:", DATA_DIR / "train_labels_v2.csv", train_labels_v2.shape)

## Özet

| karar | gerekçe |
|---|---|
| v1: iki public kaynağın 50/50 ortalaması (0.8927) | 58 vaka kaynakları ayırt edemiyor; ortalama gürültüyü azaltır |
| **v2: steven + Opus 4.5 + JEV (0.9014)** | üç bağımsız okuyucu; 07b ve 05 bununla eğitiliyor |
| Altın vakalar orijinal 0/1 etiketle, 2× ağırlık | tek gerçek sinyal |
| `pos_weight` yok | kaynakların pozitiflik oranları yanlı |
| Doğrulamada `gold_auc` ayrı raporlanır | LLM etiketine uyum ≠ radyoloğa uyum |

**Sonraki adım:** `03_dicom_onisleme.ipynb` — görüntülerin doğru okunması.